In [4]:

from langchain_community.vectorstores import FAISS 
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [5]:

pdf_loader = PyPDFLoader('/home/gz/Documents/Rebuttal AI/Agent/Logical_Fallacies_list.pdf')
docs = pdf_loader.load()




In [6]:
# pattern that exist text by name of fallacies by spliting text starts with capital letter and ends with a colon.
fallacy_label_pattern = r"\n[A-Z][^:]+:"

text_splitter = RecursiveCharacterTextSplitter(
    separators=[fallacy_label_pattern],
    is_separator_regex=True,
    chunk_size=1000, 
    chunk_overlap=0,
    keep_separator=True
)
chunks = text_splitter.split_documents(docs)

In [7]:
chunks[15].page_content

'Begging the Question (also called Petitio Principii, this term is sometimes used \ninterchangeably with Circular Reasoning): If writers assume as evidence for their \nargument the very conclusion they are attempting to prove, they engage in the fallacy of \nbegging the question. The most common form of this fallacy is when the first claim is'

### Embedding

In [8]:
embedding = OllamaEmbeddings(model='qwen3-embedding:0.6b')
embedded_docs = embedding.embed_documents(chunks)

/tmp/ipykernel_4963/2334118401.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model='qwen3-embedding:0.6b')


In [9]:
Vector_DB = FAISS.from_documents(chunks,embedding)

In [ ]:
Vector_DB.save_local('Logical_Fallacies_DB')

In [28]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
search.invoke("what is population of connecticut?")

"Jul 1, 2025 · N Data for this geographic area cannot be displayed because the number of sample cases is too small. It has a population of 3,739,160, making it the 29th most populated state in the country. The capital city is Hartford. Connecticut has a strong insurance, manufacturing, and defense industries. Use the interactive dashboard to explore estimates of Connecticut ’ s total population at the state, planning region, and town levels. Connecticut ' s population grew 1.3% from the 3.6 million people who lived there in 2010. For comparison, the population in the US grew 7.7% during that period. How has Connecticut 's population changed over the years? Connecticut 's population increased 6 out of the 12 years between year 2010 and year 2022. Census data for Connecticut (pop. 3,617,176), including age, race, sex, income, poverty, marital status, education and more. Feb 28, 2025 · Explore latest statistics on Connecticut population by year. Learn interesting facts & trends to help yo

In [24]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever()
docs = retriever.invoke("roe v wade")

print(docs[0].page_content)

Roe v. Wade, 410 U.S. 113 (1973), was a landmark decision of the U.S. Supreme Court in which the Court ruled that the Constitution of the United States protected the right to have an abortion prior to the point of fetal viability. The decision struck down many state abortion laws, and it sparked an ongoing abortion debate in the United States about whether, or to what extent, abortion should be legal, who should decide the legality of abortion, and what the role of moral and religious views in the political sphere should be. The decision also shaped debate concerning which methods the Supreme Court should use in constitutional adjudication.
The case was brought by Norma McCorvey—under the legal pseudonym "Jane Roe"—who, in 1969, became pregnant with her third child. McCorvey wanted an abortion but lived in Texas where abortion was only legal when necessary to save the mother's life. Her lawyers, Sarah Weddington and Linda Coffee, filed a lawsuit on her behalf in U.S. federal court agai

In [25]:
print(docs[1].page_content[:200])

Dobbs v. Jackson Women's Health Organization, 597 U.S. 215 (2022), is a landmark decision of the United States Supreme Court in which the court held that the United States Constitution does not confer


In [27]:
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

# Create the API wrapper
wiki_api = WikipediaAPIWrapper()

# Create the summary tool
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

# Query a topic
summary = wiki_tool.invoke("roe v wade")
print(summary)


Page: Roe v. Wade
Summary: Roe v. Wade, 410 U.S. 113 (1973), was a landmark decision of the U.S. Supreme Court in which the Court ruled that the Constitution of the United States protected the right to have an abortion prior to the point of fetal viability. The decision struck down many state abortion laws, and it sparked an ongoing abortion debate in the United States about whether, or to what extent, abortion should be legal, who should decide the legality of abortion, and what the role of moral and religious views in the political sphere should be. The decision also shaped debate concerning which methods the Supreme Court should use in constitutional adjudication.
The case was brought by Norma McCorvey—under the legal pseudonym "Jane Roe"—who, in 1969, became pregnant with her third child. McCorvey wanted an abortion but lived in Texas where abortion was only legal when necessary to save the mother's life. Her lawyers, Sarah Weddington and Linda Coffee, filed a lawsuit on her behalf

In [36]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
